In [11]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [12]:
def load_rainfall_data(rainfall_file='district wise rainfall normal.csv'):
    """Load and process rainfall data"""
    try:
        df_rainfall = pd.read_csv(rainfall_file)
        print("[+] Rainfall data loaded successfully")
        print(f"    Shape: {df_rainfall.shape}")
        print(f"    States: {df_rainfall['STATE_UT_NAME'].nunique()}")
        print(f"    Districts: {df_rainfall['DISTRICT'].nunique()}")
        return df_rainfall
    except FileNotFoundError:
        print(f"Warning: Rainfall file {rainfall_file} not found.")
        return None

In [13]:
def create_spatial_model(file_path='CGWB_data_main_cleaned.csv'):
    """Create spatial model using KNN based on lat/lon"""
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: {file_path} not found.")
        return None, None

    id_vars = ['STATE', 'DISTRICT', 'LAT', 'LON', 'SITE_TYPE', 'WLCODE']
    df_long = pd.melt(df, id_vars=id_vars, var_name='Date', value_name='Water_Level')

    df_long['Date'] = pd.to_datetime(df_long['Date'], errors='coerce')
    df_long.dropna(subset=['Water_Level', 'LAT', 'LON', 'Date'], inplace=True)

    df_recent = df_long.loc[df_long.groupby('WLCODE')['Date'].idxmax()]

    X = df_recent[['LAT', 'LON']]
    y = df_recent['Water_Level']

    knn_model = KNeighborsRegressor(n_neighbors=5, weights='distance')
    knn_model.fit(X, y)

    print("[+] Spatial model trained (KNN)")
    return knn_model, df_long

In [14]:
def get_rainfall_for_district(district, rainfall_df):
    """Get rainfall data for a specific district"""
    if rainfall_df is None:
        return None
    
    district_data = rainfall_df[rainfall_df['DISTRICT'].str.upper() == district.upper()]
    if not district_data.empty:
        return district_data.iloc[0]
    return None

In [15]:
def prepare_lstm_data_with_rainfall(yearly_avg, rainfall_series, seq_len=3):
    """Prepare LSTM data combining water level and rainfall features"""
    water_level = yearly_avg['Water_Level'].values
    
    if rainfall_series is not None:
        rainfall = rainfall_series.values if hasattr(rainfall_series, 'values') else rainfall_series
        # Normalize both to same length
        min_len = min(len(water_level), len(rainfall))
        water_level = water_level[:min_len]
        rainfall = rainfall[:min_len]
        # Stack features: [water_level, rainfall]
        combined = np.column_stack([water_level, rainfall])
        n_features = 2
    else:
        combined = water_level.reshape(-1, 1)
        n_features = 1

    X, y = [], []
    for i in range(len(combined) - seq_len):
        X.append(combined[i:i + seq_len])
        y.append(water_level[i + seq_len])

    return np.array(X), np.array(y), n_features

In [16]:
def build_cnn_lstm_multifeature(seq_len, n_features=1):
    """Build CNN+LSTM model for multifeature input"""
    model = Sequential([
        Conv1D(32, kernel_size=2, activation='relu', input_shape=(seq_len, n_features)),
        MaxPooling1D(pool_size=1),
        LSTM(64, return_sequences=False),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model

In [17]:
def cnn_lstm_predict_with_rainfall(yearly_avg, rainfall_data, future_year, seq_len=3, output_dir="outputs"):
    """Train CNN+LSTM model with rainfall features for predictions"""
    
    X, y, n_features = prepare_lstm_data_with_rainfall(yearly_avg, rainfall_data, seq_len)
    
    if len(X) == 0:
        print("Warning: Not enough data for training")
        return None, None, None
    
    scaler_water = MinMaxScaler()
    scaler_rainfall = MinMaxScaler() if n_features > 1 else None
    
    X_scaled = X.copy().astype(float)
    
    # Scale each feature separately
    X_scaled[:, :, 0] = scaler_water.fit_transform(X[:, :, 0].reshape(-1, 1)).reshape(X[:, :, 0].shape)
    
    if n_features > 1:
        X_scaled[:, :, 1] = scaler_rainfall.fit_transform(X[:, :, 1].reshape(-1, 1)).reshape(X[:, :, 1].shape)

    model = build_cnn_lstm_multifeature(seq_len, n_features)

    # Train model
    history = model.fit(X_scaled, y, epochs=100, batch_size=4, verbose=0)

    # ----- ACCURACY METRICS -----
    y_pred_train = model.predict(X_scaled, verbose=0)
    mse = mean_squared_error(y, y_pred_train)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, y_pred_train)
    r2 = r2_score(y, y_pred_train)

    print("\n[*] TRAINING ACCURACY (With Rainfall Features)")
    print(f"    MSE  : {mse:.4f}")
    print(f"    RMSE : {rmse:.4f}")
    print(f"    MAE  : {mae:.4f}")
    print(f"    R-squared : {r2:.4f}")
    print(f"    Features: {n_features} (Water Level + Rainfall)" if n_features > 1 else f"    Features: {n_features}")

    # ----- TRAINING LOSS PLOT -----
    os.makedirs(output_dir, exist_ok=True)

    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], linewidth=2)
    plt.title("CNN + LSTM Model Loss Curve (With Rainfall Feature)", fontsize=13, fontweight='bold')
    plt.xlabel("Epochs", fontsize=11)
    plt.ylabel("Loss (MSE)", fontsize=11)
    plt.grid(True, alpha=0.3)
    loss_plot_path = os.path.join(output_dir, "training_loss_with_rainfall.png")
    plt.savefig(loss_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"\n[+] Loss plot saved: {loss_plot_path}")

    # ----- FUTURE PREDICTIONS -----
    future_predictions = {}
    last_seq = X[-1]
    last_year = yearly_avg["Year"].max()

    for year in range(last_year + 1, future_year + 1):
        seq_scaled = last_seq.copy().astype(float)
        seq_scaled[:, 0] = scaler_water.transform(last_seq[:, 0].reshape(-1, 1)).flatten()
        if n_features > 1:
            seq_scaled[:, 1] = scaler_rainfall.transform(last_seq[:, 1].reshape(-1, 1)).flatten()
        
        seq_scaled = seq_scaled.reshape(1, seq_len, n_features)
        next_pred = model.predict(seq_scaled, verbose=0)[0][0]

        future_predictions[year] = next_pred
        
        # Update sequence with prediction and estimated rainfall
        if n_features > 1:
            estimated_rainfall = np.mean(last_seq[1:, 1])
            last_seq = np.vstack([last_seq[1:], [next_pred, estimated_rainfall]])
        else:
            last_seq = np.vstack([last_seq[1:], [[next_pred]]])

    return future_predictions, loss_plot_path, (mse, rmse, mae, r2)

In [18]:
def predict_until_year_with_rainfall(latitude, longitude, district, future_year, model, df_long, rainfall_df, output_dir="outputs"):
    """Main prediction function integrating rainfall data"""
    if model is None or df_long is None:
        return "Model unavailable."

    df = df_long

    input_data = pd.DataFrame([[latitude, longitude]], columns=['LAT', 'LON'])
    distances, indices = model.kneighbors(input_data, n_neighbors=1)

    nearest_lat, nearest_lon = model._fit_X[indices[0][0]]
    nearest_well = df[(df['LAT'] == nearest_lat) & (df['LON'] == nearest_lon)]['WLCODE'].iloc[0]

    ts = df[df['WLCODE'] == nearest_well].copy()
    ts["Year"] = ts["Date"].dt.year
    yearly_avg = ts.groupby("Year")["Water_Level"].mean().reset_index()
    
    print(f"\n[*] Historical data for well {nearest_well}:")
    print(f"    Time period: {yearly_avg['Year'].min()} - {yearly_avg['Year'].max()}")
    print(f"    Records: {len(yearly_avg)}")

    # Get rainfall data for the district
    rainfall_yearly = None
    if district and rainfall_df is not None:
        rainfall_data = get_rainfall_for_district(district, rainfall_df)
        if rainfall_data is not None:
            # Create yearly rainfall series (using annual rainfall)
            rainfall_yearly = pd.Series([rainfall_data['ANNUAL']] * len(yearly_avg), index=yearly_avg.index)
            print(f"[+] Using annual rainfall data for district: {district}")
            print(f"    Annual rainfall: {rainfall_data['ANNUAL']:.2f} mm")
        else:
            print(f"[-] District '{district}' not found in rainfall database")
            print(f"    Proceeding with water level data only")

    # Predictions with rainfall + accuracy + training loss image
    predictions, loss_img, scores = cnn_lstm_predict_with_rainfall(yearly_avg, rainfall_yearly, future_year)

    if predictions is None:
        return nearest_well, {}, "", "", scores

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.plot(yearly_avg["Year"], yearly_avg["Water_Level"], marker="o", 
             label="Historical Data", linewidth=2, markersize=6)
    plt.plot(list(predictions.keys()), list(predictions.values()), marker="s",
             linestyle="--", color="red", label="Predicted (CNN+LSTM+Rainfall)", 
             linewidth=2, markersize=6)
    plt.xlabel("Year", fontsize=12, fontweight='bold')
    plt.ylabel("Groundwater Level (meters)", fontsize=12, fontweight='bold')
    title = f"Groundwater Level Prediction - Well {nearest_well}\nUsing CNN+LSTM with Rainfall Feature"
    if district:
        title += f" | District: {district}"
    plt.title(title, fontsize=13, fontweight='bold')
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)

    plot_path = os.path.join(output_dir, f"prediction_rainfall_{nearest_well}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[+] Prediction plot saved: {plot_path}")

    return nearest_well, predictions, plot_path, loss_img, scores

In [19]:
# Initialize models
print("=" * 70)
print("GROUNDWATER LEVEL PREDICTION WITH RAINFALL INTEGRATION")
print("=" * 70)

spatial_model, df_long = create_spatial_model('CGWB_data_main_cleaned.csv')
rainfall_df = load_rainfall_data('district wise rainfall normal.csv')

print("\n" + "=" * 70)

GROUNDWATER LEVEL PREDICTION WITH RAINFALL INTEGRATION
[+] Spatial model trained (KNN)
[+] Rainfall data loaded successfully
    Shape: (641, 19)
    States: 35
    Districts: 637



In [20]:
# Get user input and run prediction
if spatial_model:
    print("\n[?] Enter location coordinates and district name:")
    lat = float(input("    Latitude: "))
    lon = float(input("    Longitude: "))
    district = input("    District name: ").strip()
    year = int(input("    Predict up to year: "))

    print("\n[*] Processing... this may take a moment")
    well, preds, plot_file, loss_file, acc = predict_until_year_with_rainfall(
        lat, lon, district, year, spatial_model, df_long, rainfall_df
    )

    print("\n" + "=" * 70)
    print("PREDICTION RESULTS")
    print("=" * 70)
    print(f"\n[*] Nearest Well ID: {well}")
    if district:
        print(f"[*] District: {district}")

    print(f"\n[*] Groundwater Level Predictions (in meters):")
    for yr, lvl in preds.items():
        print(f"    Year {yr}: {lvl:.2f} m")

    print(f"\n[*] MODEL PERFORMANCE METRICS:")
    print(f"    Mean Squared Error (MSE)  : {acc[0]:.4f}")
    print(f"    Root Mean Squared Error   : {acc[1]:.4f}")
    print(f"    Mean Absolute Error (MAE) : {acc[2]:.4f}")
    print(f"    R-squared (R2)            : {acc[3]:.4f}")

    print(f"\n[+] Output Files:")
    print(f"    Prediction plot: {plot_file}")
    print(f"    Loss curve plot: {loss_file}")
    print("\n" + "=" * 70)
else:
    print("[!] Error: Failed to initialize model")


[?] Enter location coordinates and district name:

[*] Processing... this may take a moment

[*] Historical data for well W30848:
    Time period: 1996 - 2017
    Records: 22
[+] Using annual rainfall data for district: Ghaziabad
    Annual rainfall: 766.30 mm

[*] TRAINING ACCURACY (With Rainfall Features)
    MSE  : 0.1778
    RMSE : 0.4216
    MAE  : 0.2593
    R-squared : 0.6140
    Features: 2 (Water Level + Rainfall)

[+] Loss plot saved: outputs\training_loss_with_rainfall.png
[+] Prediction plot saved: outputs\prediction_rainfall_W30848.png

PREDICTION RESULTS

[*] Nearest Well ID: W30848
[*] District: Ghaziabad

[*] Groundwater Level Predictions (in meters):
    Year 2018: 2.48 m
    Year 2019: 2.18 m
    Year 2020: 1.96 m
    Year 2021: 1.85 m
    Year 2022: 1.79 m
    Year 2023: 1.81 m
    Year 2024: 1.86 m
    Year 2025: 1.89 m
    Year 2026: 1.90 m
    Year 2027: 1.88 m
    Year 2028: 1.86 m
    Year 2029: 1.86 m
    Year 2030: 1.86 m
    Year 2031: 1.87 m
    Year 2032: 